**DEMONSTRATION:**

- Langchain Agents [document link](https://docs.langchain.com/oss/python/langchain/agents)
- Langchain Models [document link](https://docs.langchain.com/oss/python/langchain/models)
- 

In [47]:
import os
import json
import sqlite3
from dotenv import load_dotenv

import langchain
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.callbacks import BaseCallbackHandler

from langchain.chat_models import init_chat_model
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [48]:
langchain.__version__

'1.2.6'

In [2]:
load_dotenv()
HUGGINGFACE_API_TOKEN = os.getenv("HUGGINGFACE_API_TOKEN")

In [3]:
MODEL_CONFIGS = {
    "qwen": {
        "name": "Qwen/Qwen2.5-72B-Instruct",
        "description": "Best overall - 72B params, excellent reasoning",
        "max_tokens": 4096,
    },
    "mixtral": {
        "name": "mistralai/Mixtral-8x7B-Instruct-v0.1",
        "description": "Fast and good - 47B effective params",
        "max_tokens": 4096,
    },
    "llama": {
        "name": "meta-llama/Meta-Llama-3.1-70B-Instruct",
        "description": "Meta's best - 70B params",
        "max_tokens": 4096,
    },
    "command-r": {
        "name": "CohereForAI/c4ai-command-r-plus",
        "description": "Excellent for tools and RAG - 104B params",
        "max_tokens": 4096,
    },
    "phi": {
        "name": "microsoft/Phi-3-medium-4k-instruct",
        "description": "Smaller but capable - 14B params",
        "max_tokens": 4096,
    }
}

In [ ]:
# ### This loads the model locally, Does Not use the hosted Hugging Face Inference API.

# model = init_chat_model(
#     "microsoft/Phi-3-medium-4k-instruct",
#     model_provider="huggingface",
#     temperature=0.7,
#     max_tokens=1024,
# )
# response = model.invoke("Why do parrots talk?")

**NOTE:**

- `init_chat_model` loads the model locally, Does Not use the hosted Hugging Face Inference API.
- For hosted inference instead of local weights:
    - Need to use `HuggingFaceEndpoint` explicitly (wraped in `ChatHuggingFace` API)

In [4]:
SELECTED_MODEL = "qwen"
model_config = MODEL_CONFIGS[SELECTED_MODEL]

In [5]:
base_llm = HuggingFaceEndpoint(
    repo_id=model_config['name'],
    huggingfacehub_api_token=HUGGINGFACE_API_TOKEN,
    max_new_tokens=model_config['max_tokens'],
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.1,
)
chat_model = ChatHuggingFace(llm=base_llm)

In [37]:
response = chat_model.invoke("What is LangChain?")

In [39]:
# response
# print(response.content)
dict(response)

{'content': "LangChain is a framework designed to help developers build applications that leverage large language models (LLMs) more effectively. It provides a set of tools, libraries, and best practices to facilitate the integration of LLMs into various applications, making it easier to handle tasks such as natural language processing, text generation, and conversational AI.\n\nKey features of LangChain include:\n\n1. **Modular Architecture**: LangChain is built with a modular design, allowing developers to mix and match different components to suit their specific needs. This includes various language models, data sources, and output formats.\n\n2. **Integration with Multiple Models**: It supports integration with a wide range of LLMs, including those from different providers like OpenAI, Anthropic, and others.\n\n3. **Data Handling**: LangChain provides tools for preprocessing and postprocessing data, making it easier to prepare inputs for LLMs and handle their outputs.\n\n4. **Custo

In [49]:
for chunk in chat_model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

|Par|rots| have| colorful feathers for several| reasons, which are| primarily related to their| survival and social interactions|:

1. **|Camouflage**: Despite| the vibrant colors,| parrots can blend| into their natural habitats|, such as dense| forests. The bright| colors can help them| match the varied and| often colorful environment of| tropical and subtropical| regions.

2.| **Mate Attraction|**: Bright and colorful| feathers can be a| sign of good health| and genetic fitness.| In many species,| males with more vibrant| plumage are more| attractive to females,| increasing their chances of| mating and passing on| their genes.

3|. **Social Sign|aling**: Colors can| also play a role| in communication within par|rot flocks.| Different colors or patterns| can help individuals recognize| each other, establish| dominance, and maintain| social bonds.

4|. **Therm|oregulation**:| Some studies suggest that| certain colors can help| with temperature regulation.| For example, darker| colors c

In [48]:
chat_model.invoke("Why do parrots have colorful feathers?")

AIMessage(content='Parrots have colorful feathers for several reasons, which are primarily related to their survival and social interactions:\n\n1. **Camouflage**: Despite the vibrant colors, parrots can blend into their natural habitats, such as rainforests, where the environment is often lush and colorful. This helps them avoid predators.\n\n2. **Mate Attraction**: Bright and varied colors can be attractive to potential mates. In many species, the more vibrant and healthy-looking the feathers, the more likely the bird is to attract a mate. This is because bright colors can indicate good health and genetic fitness.\n\n3. **Social Signaling**: Colors can also play a role in communication within parrot flocks. Different colors or patterns can help individuals recognize each other, establish dominance, or convey other social signals.\n\n4. **Thermoregulation**: Some studies suggest that certain colors can help with temperature regulation. For example, darker colors can absorb more heat, 

**TOOL CALLS**\
**=============**

- `bind_tools()` is: Tool-aware model, NOT a full agent. 
- `create_agent( with passing tools as params)`: This creates a FULL agent runtime.

**What happens internally**
- The model receives:
    - tool names
    - descriptions
    - argument schemas
- The tool is NOT automatically executed.
- We must execute it manually.

**What the agent does automatically**\
The agent:
1. Sends tool schemas to model
2. Detects tool calls
3. Executes tools automatically
4. Feeds tool results back to model
5. Continues reasoning
6. Returns final answer

---

Important distinction
bind_tools()

YOU control orchestration.

create_agent()

LANGCHAIN controls orchestration.

Mental model
bind_tools

Low-level primitive.

Equivalent to:

"Tool calling enabled"
create_agent

High-level autonomous workflow engine.

Equivalent to:

"AI assistant capable of using tools"

In [50]:
base_llm = HuggingFaceEndpoint(
    repo_id=model_config['name'],
    huggingfacehub_api_token=HUGGINGFACE_API_TOKEN,
    max_new_tokens=model_config['max_tokens'],
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.1,
)
chat_model = ChatHuggingFace(llm=base_llm)

In [51]:
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools = chat_model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")

In [54]:
# dict(response)

In [55]:
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


In [56]:
# ## TOOL EXECUTION LOOP AND FINAL RESPONSE
# =========================================

# # Bind (potentially multiple) tools to the model
# model_with_tools = chat_model.bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
print(f"Tool call after invoking the prompt:\n{messages}")

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

Tool call after invoking the prompt:
[{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"location": "Boston"}', 'name': 'get_weather', 'description': None}, 'id': 'call_714775ec687a405b853d4e', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 167, 'total_tokens': 184}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e7372-b2e8-7cc3-9c28-0275fd31e241-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_714775ec687a405b853d4e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 167, 'output_tokens': 17, 'total_tokens': 184})]
The weather in Boston is sunny.


In [6]:
base_llm = HuggingFaceEndpoint(
    repo_id=model_config['name'],
    huggingfacehub_api_token=HUGGINGFACE_API_TOKEN,
    max_new_tokens=model_config['max_tokens'],
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.1,
)
chat_model = ChatHuggingFace(llm=base_llm)

In [7]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))
tools = [calculator]

In [8]:
agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt="You are a helpful assistant."
)

In [9]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 45 * 12?"
        }
    ]
})
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='What is 45 * 12?', additional_kwargs={}, response_metadata={}, id='d4741e5f-0895-4d17-a88f-fc9d20a7a109'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"expression": "45 * 12"}', 'name': 'calculator', 'description': None}, 'id': 'call_QgnKY4ynXyZuM2PKQA0xCvRo', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 302, 'total_tokens': 326}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e81a0-c5ad-7232-8170-a0d01a12e2b4-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '45 * 12'}, 'id': 'call_QgnKY4ynXyZuM2PKQA0xCvRo', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 302, 'output_tokens': 24, 'total_tokens': 326}),
              ToolMessage(content='540', name='calculator', id='f27b0c76-d1dd-4eb2-84d9-df6a032cc423', tool_call_

**Experiment: Parallel Tool Calls**

- For all independent questions, parallel tool calls is working. 

In [66]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 45 * 12? What is 45 + 12?"
        }
    ]
})
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='What is 45 * 12? What is 45 + 12?', additional_kwargs={}, response_metadata={}, id='66b23152-c5f5-464a-ba91-edf7d45cac38'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"expression": "45 * 12"}', 'name': 'calculator', 'description': None}, 'id': 'call_mE7PV1i3jZUnH6DM3uML3vaU', 'type': 'function'}, {'function': {'arguments': '{"expression": "45 + 12"}', 'name': 'calculator', 'description': None}, 'id': 'call_Nz2KLO1iM7hwMFeXT7KsIaB1', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 312, 'total_tokens': 360}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e778a-e721-7263-aee9-ccc5ebd609e1-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '45 * 12'}, 'id': 'call_mE7PV1i3jZUnH6DM3uML3vaU', 'type': 'tool_call'}, {'name': 'calculator', 'args': {'expressi

- For mix of dep + independent question, parallel tool calls is Not working. 

In [69]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 10 * 10? What is 10 + 10? What is sum up result of the previous two results?"
        }
    ]
})
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='What is 10 * 10? What is 10 + 10? What is sum up result of the previous two results?', additional_kwargs={}, response_metadata={}, id='b2d22b1e-3dd7-43b1-8656-1d015a1bdc59'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"expression": "10 * 10"}', 'name': 'calculator', 'description': None}, 'id': 'call_11dbbae4d6384117b3617e', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 191, 'total_tokens': 212}, 'model_name': 'Qwen/Qwen2.5-72B-Instruct', 'system_fingerprint': '', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e779b-2426-7750-9a31-41fb2bab52c5-0', tool_calls=[{'name': 'calculator', 'args': {'expression': '10 * 10'}, 'id': 'call_11dbbae4d6384117b3617e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 191, 'output_tokens': 21, 'total_tokens': 212}),
              ToolMessage(content='100', name='ca

**Summary Till Now:**

- Model call: `init_chat_model` VS `HuggingFaceEndpoint` wraped in `ChatHuggingFace` API.
- Agent Invoke: Normal response VS Streaming responponse from model 
- Tools and Tool calls
- Sequential VS Parallel Tool Calls

**NEXT**

- Structured Output
- 

In [36]:
base_llm = HuggingFaceEndpoint(
    repo_id=model_config['name'],
    huggingfacehub_api_token=HUGGINGFACE_API_TOKEN,
    max_new_tokens=model_config['max_tokens'],
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.1,
)
chat_model = ChatHuggingFace(llm=base_llm)

In [37]:
# print(chat_model.profile)
# vars(chat_model)

In [38]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))
tools = [calculator]

In [39]:
agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt="You are a helpful assistant."
)

In [40]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Provide details about the movie Inception"
        }
    ]
})

In [41]:
from pprint import pprint
# pprint(response)
print(response['messages'][-1].content)

I don't have the ability to call a function that provides movie details, but I can give you a brief overview of the movie "Inception" based on my knowledge:

"Inception" is a 2010 science fiction action film written, co-produced, and directed by Christopher Nolan. The film stars Leonardo DiCaprio as a professional thief who steals corporate secrets through the use of dream-sharing technology. He is given the inverse task of planting an idea into the mind of a CEO, a process called inception. The film also features a ensemble cast including Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe, Dileep Rao, Cillian Murphy, Tom Berenger, and Marion Cotillard.

If you need more specific details, such as the release date, box office performance, or critical reception, let me know!


In [31]:
# ## THIS APPROACH DIDN'T WORK..
# from pydantic import BaseModel
# class Movie(BaseModel):
#     title: str
#     year: int
#     director: str
#     rating: float

In [42]:
sysprompt = """
You are a helpful assistant.

Always return valid JSON in this format:

{
  "title": string,
  "year": integer,
  "director": string,
  "rating": number
}

Return ONLY JSON.
"""

In [43]:
agent = create_agent(
    model=chat_model,
    tools=tools,
    system_prompt=sysprompt,
    # response_format=Movie  # ## [[ THIS DIDN'T WORK ]]
)

In [44]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Provide details about the movie Inception"
        }
    ]
})

In [45]:
from pprint import pprint
# pprint(response)
print(response['messages'][-1].content)

{
  "title": "Inception",
  "year": 2010,
  "director": "Christopher Nolan",
  "rating": 8.8
}


In [50]:
print(base_llm.repo_id)
print(type(base_llm))

Qwen/Qwen2.5-72B-Instruct
<class 'langchain_huggingface.llms.huggingface_endpoint.HuggingFaceEndpoint'>
